# 02 — Exploratory Data Analysis

**Dataset:** `merged_cleaned_sold_with_josiah.csv`  
**Rows:** 55,133 | **Columns:** 75  
**Target:** `sold_price`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/merged_cleaned_sold_with_josiah.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

## 1. Dataset Overview

In [ ]:
df.info(verbose=False, memory_usage='deep')

In [ ]:
# Key columns we care about for modeling
KEY_COLS = ['listing_price', 'sold_price', 'bedrooms', 'bathrooms',
            'square_feet_num', 'lot_size_sqft', 'taxes', 'median_income',
            'parking_spaces', 'city', 'property_type', 'house_age',
            'number_of_storeys', 'total_rooms', 'bath_bed_ratio',
            'listing_year', 'listing_month', 'days_since_sold']

df[KEY_COLS].describe(include='all')

## 2. Missing Values

In [ ]:
missing = (df[KEY_COLS].isnull().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing.index, missing.values, color='steelblue')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values in Key Columns')
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png')
plt.show()

## 3. Target Variable — Sold Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw distribution
axes[0].hist(df['sold_price'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Sold Price Distribution')
axes[0].set_xlabel('Sold Price (CAD)')
axes[0].set_ylabel('Count')

# Log distribution
axes[1].hist(np.log1p(df['sold_price']), bins=80, color='coral', edgecolor='white')
axes[1].set_title('Log(Sold Price) Distribution')
axes[1].set_xlabel('log(Sold Price)')

plt.tight_layout()
plt.savefig('../reports/figures/sold_price_distribution.png')
plt.show()

print(df['sold_price'].describe().apply(lambda x: f'${x:,.0f}'))

## 4. Listing Price vs Sold Price

In [ ]:
df['price_diff_pct'] = (df['sold_price'] - df['listing_price']) / df['listing_price'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter
sample = df.sample(min(5000, len(df)), random_state=42)
axes[0].scatter(sample['listing_price'], sample['sold_price'], alpha=0.2, s=5, color='steelblue')
lims = [0, df[['listing_price','sold_price']].max().max()]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Listing Price (CAD)')
axes[0].set_ylabel('Sold Price (CAD)')
axes[0].set_title('Listing vs Sold Price')

# Price diff distribution
clipped = df['price_diff_pct'].clip(-50, 50)
axes[1].hist(clipped, bins=60, color='teal', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('% Difference: Sold vs Listed (clipped ±50%)')
axes[1].set_xlabel('% Change')

plt.tight_layout()
plt.savefig('../reports/figures/listing_vs_sold.png')
plt.show()

print(f"Sold above listing: {(df['price_diff_pct'] > 0).mean()*100:.1f}%")
print(f"Median price diff:  {df['price_diff_pct'].median():.2f}%")

## 5. Sold Price by City

In [ ]:
top_cities = df['city'].value_counts().head(10).index
city_df = df[df['city'].isin(top_cities)]

order = city_df.groupby('city')['sold_price'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=city_df, x='city', y='sold_price', order=order, ax=ax,
            showfliers=False, palette='Blues_d')
ax.set_title('Sold Price by City (top 10, outliers hidden)')
ax.set_xlabel('')
ax.set_ylabel('Sold Price (CAD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/price_by_city.png')
plt.show()

## 6. Sold Price by Property Type

In [ ]:
top_types = df['property_type'].value_counts().head(8).index
type_df = df[df['property_type'].isin(top_types)]
order = type_df.groupby('property_type')['sold_price'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=type_df, x='property_type', y='sold_price', order=order,
            showfliers=False, palette='Oranges_d', ax=ax)
ax.set_title('Sold Price by Property Type (outliers hidden)')
ax.set_xlabel('')
ax.set_ylabel('Sold Price (CAD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/price_by_type.png')
plt.show()

## 7. Numeric Feature Distributions

In [ ]:
num_cols = ['bedrooms', 'bathrooms', 'square_feet_num', 'lot_size_sqft',
            'parking_spaces', 'total_rooms', 'taxes', 'median_income']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), num_cols):
    data = df[col].dropna()
    # Clip extreme outliers for readability
    data = data[data <= data.quantile(0.99)]
    ax.hist(data, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Numeric Feature Distributions (clipped at 99th percentile)', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png')
plt.show()

## 8. Correlation Heatmap

In [ ]:
corr_cols = ['sold_price', 'listing_price', 'bedrooms', 'bathrooms',
             'square_feet_num', 'lot_size_sqft', 'taxes', 'median_income',
             'parking_spaces', 'total_rooms', 'bath_bed_ratio', 'days_since_sold']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix — Key Features vs Sold Price')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png')
plt.show()

## 9. Bedrooms & Bathrooms vs Sold Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, color in zip(axes, ['bedrooms', 'bathrooms'], ['steelblue', 'coral']):
    valid = df[df[col].between(1, 8)]
    order = sorted(valid[col].unique())
    sns.boxplot(data=valid, x=col, y='sold_price', order=order,
                showfliers=False, color=color, ax=ax)
    ax.set_title(f'Sold Price by {col.capitalize()}')
    ax.set_xlabel(col.capitalize())
    ax.set_ylabel('Sold Price (CAD)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/figures/beds_baths_vs_price.png')
plt.show()

## 10. Listings Over Time

In [ ]:
monthly = df.groupby(['listing_year', 'listing_month']).agg(
    count=('sold_price', 'count'),
    median_price=('sold_price', 'median')
).reset_index()
monthly['period'] = pd.to_datetime(monthly[['listing_year','listing_month']].rename(
    columns={'listing_year':'year','listing_month':'month'}).assign(day=1))
monthly = monthly.sort_values('period')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].bar(monthly['period'], monthly['count'], color='steelblue', width=20)
axes[0].set_title('Number of Sales per Month')
axes[0].set_ylabel('Count')

axes[1].plot(monthly['period'], monthly['median_price'], color='coral', marker='o', markersize=3)
axes[1].set_title('Median Sold Price per Month')
axes[1].set_ylabel('Median Sold Price (CAD)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/figures/listings_over_time.png')
plt.show()

## 11. Key EDA Findings Summary

In [ ]:
print('=== EDA Summary ===')
print(f"Total listings:          {len(df):,}")
print(f"Median sold price:       ${df['sold_price'].median():,.0f}")
print(f"Mean sold price:         ${df['sold_price'].mean():,.0f}")
print(f"Sold above listing:      {(df['price_diff_pct'] > 0).mean()*100:.1f}%")
print(f"Median price diff:       {df['price_diff_pct'].median():.2f}%")
print(f"Most common city:        {df['city'].mode()[0]}")
print(f"Most common type:        {df['property_type'].mode()[0]}")
print(f"Corr listing→sold:       {df['listing_price'].corr(df['sold_price']):.4f}")
print(f"Corr sqft→sold:          {df['square_feet_num'].corr(df['sold_price']):.4f}")
print(f"Corr bedrooms→sold:      {df['bedrooms'].corr(df['sold_price']):.4f}")